# End-to-End MLP Workflow

This notebook is a fully worked example of how to use `portfolio_toolkit` with a basic MLP-style forecasting model.

It covers the full notebook-first workflow:

1. load a shared dataset
2. add built-in toolkit features
3. engineer new custom notebook-local features
4. build forward return / alpha / volatility targets
5. split into train / validation / test using the shared split rules
6. define and train a small PyTorch MLP regressor
7. emit a standardized prediction table
8. turn predictions into a `PortfolioWeights` object
9. run the shared backtest
10. write reports and artifacts
11. log everything to MLflow

This is intentionally simple and heavily commented. The idea is that a teammate can copy this notebook, replace the model body, keep the shared data/evaluation layer, and still be comparable to everyone else.

## Running This Notebook In Colab

If you want to run this notebook in Google Colab, start by cloning the repo into the Colab session and installing the toolkit in editable mode.

Steps:

1. Set `REPO_URL` below to your GitHub repo URL.
2. Run the bootstrap cell once.
3. After that, the rest of the notebook can import `portfolio_toolkit` normally.

If you are running locally, the same cell will automatically fall back to the repository on your machine.


In [1]:
# Colab / local bootstrap cell
# - In Colab: clone the repo, install the package, and point repo_root at /content/...
# - Locally: just point repo_root at this repository on disk

import sys
from pathlib import Path
 
IN_COLAB = "google.colab" in sys.modules
 
if IN_COLAB:
    REPO_URL = "https://github.com/<your-user-or-org>/<your-repo>.git"  # replace with real URL
    REPO_DIR = "/content/Portfolio-Optimizer"
    if "<your-user-or-org>" in REPO_URL or "<your-repo>" in REPO_URL:
        raise ValueError("Set REPO_URL to your real GitHub repository URL before running in Colab.")
    import subprocess
    subprocess.run(["rm", "-rf", REPO_DIR])
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR])
    import os; os.chdir(REPO_DIR)
    subprocess.run(["pip", "install", "-e", ".[dev]"])
    repo_root = Path(REPO_DIR).resolve()
else:
    repo_root = Path(repo_root).resolve() if "repo_root" in globals() else Path("../../").resolve()
 
print("repo_root =", repo_root)


repo_root = /Users/adamthorne/Desktop/Portfolio/Portfolio-Optimizer


In [ ]:
import sys, pathlib
repo_root = pathlib.Path("/Users/hannahlee/Documents/MLSN/Portfolio-Optimization-Lib")
sys.path.append(str(repo_root / "src"))
from portfolio_toolkit import init_mlflow

In [2]:
#ml flow setup?
from portfolio_toolkit import init_mlflow

mlflow_layout = init_mlflow(repo_root)
print(mlflow_layout)

/Users/adamthorne/.pyenv/versions/3.12.7/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'tracking_uri': 'https://adams-macbook-pro.tail5ddc35.ts.net', 'backend_store_uri': 'https://adams-macbook-pro.tail5ddc35.ts.net', 'artifact_root': '', 'db_path': ''}


## What This Example Is Doing

This notebook uses `shared_set_2` by default instead of `shared_set_1`.

Why:

- `shared_set_1` is the full S&P 500 universe, so the first download is much larger.
- `shared_set_2` is smaller and faster for a first MLP example.

Once you understand the pattern, you can switch `dataset_name` to `shared_set_1` and run the exact same workflow over a much broader universe.

In plain English:
“What data am I using, what am I predicting, and where do I save results?”

In [14]:
from pathlib import Path
import numpy as np
import pandas as pd
 
from portfolio_toolkit import (
    build_features,
    build_metrics,
    get_dataset_spec,
    init_mlflow,
    load_prices,
    log_backtest,
    log_portfolio,
    log_predictions,
    make_forward_alpha_target,
    make_forward_realized_vol_target,
    make_forward_return_target,
    slice_split,
    split_dates,
    start_run,
    validate_prediction_frame,
    validate_weights_frame,
    weights_from_predictions_risk_adjusted,
    backtest_weights,
    write_backtest_artifacts,
)

# ---------------------------------------------------------------------
# Basic run configuration.
# ---------------------------------------------------------------------
# repo_root:
#   Where the toolkit config files, cache, MLflow folder, and runs folder live.
# dataset_name:
#   Which shared ticker universe + split rules we want to use.
# horizon:
#   Our prediction horizon in trading days.
# output_dir:
#   Where this notebook will write artifacts like metrics and QuantStats.
# ---------------------------------------------------------------------

repo_root     = Path(repo_root).resolve() if "repo_root" in globals() else Path("../../").resolve()
dataset_name  = "secret_set_1"
model_name    = "autoencoder_mlp_downstream"          # was: torch_mlp_forecast_example
horizon       = 5
output_dir    = repo_root / "runs" / "autoencoder_downstream"  # was: mlp_end_to_end_workflow
output_dir.mkdir(parents=True, exist_ok=True)
 
# FIX: single source of truth for hyperparameters — used in training AND mlflow logging.
AE_EPOCHS    = 25
MLP_EPOCHS   = 30
BATCH_SIZE   = 1024
LATENT_DIM   = 16
LEARNING_RATE = 1e-3
PATIENCE     = 5   # early stopping patience (epochs without val improvement)
 
spec   = get_dataset_spec(dataset_name, repo_root=repo_root)
splits = split_dates(dataset_name, repo_root=repo_root)
 
print("Dataset preset:",       dataset_name)
print("Dataset display name:", spec.name)
print("Tickers modeled:",      len(spec.tickers))
print("Benchmark ticker:",     spec.benchmark_ticker)
print("Train/Val/Test windows:", splits)

Dataset preset: secret_set_1
Dataset display name: XLE_test
Tickers modeled: 22
Benchmark ticker: XLE
Train/Val/Test windows: {'train': (Timestamp('2014-01-02 00:00:00'), Timestamp('2019-12-31 00:00:00')), 'val': (Timestamp('2020-01-02 00:00:00'), Timestamp('2021-12-31 00:00:00')), 'test': (Timestamp('2022-01-03 00:00:00'), Timestamp('2025-12-31 00:00:00'))}


## 1. Load Shared Price Data

The toolkit's `load_prices(...)` function is the shared data entrypoint.

What it does:

- reads the selected dataset preset from `configs/datasets.toml`
- downloads daily OHLCV data with `yfinance` if it is not already cached
- always includes `SPY` as the benchmark series
- validates and normalizes the dataframe before returning it

This is one of the main standardization points in the repo: everyone starts from the same dataset preset and the same split boundaries.

In plain English:
“Give me clean stock price data everyone on the team uses.”


In [15]:
prices = load_prices(dataset_name, repo_root=repo_root)

print('Price frame shape:', prices.shape)
print('Date range:', prices['date'].min(), '->', prices['date'].max())
print('Number of unique tickers in price frame:', prices['ticker'].nunique())
display(prices.head())

Price frame shape: (67625, 8)
Date range: 2014-01-02 00:00:00 -> 2025-12-31 00:00:00
Number of unique tickers in price frame: 23


,date,ticker,open,high,low,close,adj_close,volume
0,2014-01-02,APA,85.820000,85.970001,85.160004,85.480003,63.762150,2565400
1,2014-01-03,APA,85.800003,86.730003,85.379997,85.540001,63.806904,2259400
2,2014-01-06,APA,85.760002,86.470001,85.120003,86.309998,64.381264,2316000
3,2014-01-07,APA,86.419998,87.910004,86.129997,87.900002,65.567291,2812300
4,2014-01-08,APA,87.559998,87.580002,86.360001,86.650002,64.634888,2712800


## 2. Add Built-In Toolkit Features

We start with a moderate built-in feature set.

These are all created by the shared library, which means other teammates can use the same starting point if they want:

- momentum
- volatility
- RSI
- moving-average distance
- volume z-score
- benchmark-relative return
- intraday range and beta vs SPY

This is a good example of the intended workflow:

- shared features for consistency and speed
- custom notebook-local features on top when you want to experiment

In plain English:
“Turn raw prices into useful signals the model can learn from.” For example, raw price to momentum, volatility, and RSI


In [5]:
import random #pick random seed for reproducibility
import torch
random.seed(99)
np.random.seed(99)
torch.manual_seed(99)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(99)
torch.backends.cudnn.deterministic = True

In [16]:
base_feature_names = [
    'return_5d',
    'return_20d',
    'vol_20d',
    'momentum_20d',
    'momentum_60d',
    'rsi_14',
    'price_to_sma_20d',
    'price_to_sma_50d',
    'volume_zscore_20d',
    'excess_return_20d_vs_spy',
    'intraday_range',
    'beta_20d_spy',
]

base_features = build_features(prices, feature_names=base_feature_names)
print('Base feature frame shape:', base_features.shape)
display(base_features)

Base feature frame shape: (67625, 14)


,date,ticker,return_5d,return_20d,vol_20d,momentum_20d,momentum_60d,rsi_14,price_to_sma_20d,price_to_sma_50d,volume_zscore_20d,excess_return_20d_vs_spy,intraday_range,beta_20d_spy
0,2014-01-02,APA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.009476,NaN
1,2014-01-03,APA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.015782,NaN
2,2014-01-06,APA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.015641,NaN
3,2014-01-07,APA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.020250,NaN
4,2014-01-08,APA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.014080,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67620,2025-12-24,XOM,0.015416,0.041132,0.012024,0.041132,0.066632,56.396080,0.016290,0.029031,-1.259918,NaN,0.007801,NaN
67621,2025-12-26,XOM,0.022052,0.037815,0.012041,0.037815,0.072880,58.148373,0.013478,0.026581,-0.972248,NaN,0.008564,NaN
67622,2025-12-29,XOM,0.032908,0.039769,0.012112,0.039769,0.092499,63.680082,0.023553,0.036881,-0.177936,NaN,0.015764,NaN
67623,2025-12-30,XOM,0.024037,0.037383,0.012084,0.037383,0.077594,59.244257,0.025561,0.039100,-0.637130,NaN,0.009670,NaN


## 3. Add New Custom Features In The Notebook

This is where developers keep their freedom.

The toolkit does **not** force everyone to only use built-in features. A normal workflow is:

1. build a shared baseline feature set
2. add experimental features locally in the notebook
3. keep using the shared validation / portfolio / backtest layer afterward

Below we add a few simple handcrafted features:

- `mom_vol_ratio`
  A momentum-to-volatility ratio. This is a quick-and-dirty risk-adjusted trend signal.
- `trend_spread`
  The gap between short-term and medium-term trend distance.
- `quality_signal`
  A simple benchmark-relative momentum signal penalized by volatility.
- `range_volume_interaction`
  A rough interaction term between price range expansion and unusual volume.

In plain English:
“Add your own ideas on top of the default signals.”


In [17]:
frame = base_features.copy()

# Add a very small constant anywhere we divide so we do not create infinities.
eps = 1e-6

frame['mom_vol_ratio'] = frame['momentum_20d'] / (frame['vol_20d'].abs() + eps)
frame['trend_spread'] = frame['price_to_sma_20d'] - frame['price_to_sma_50d']
frame['quality_signal'] = frame['excess_return_20d_vs_spy'] - 0.5 * frame['vol_20d']
frame['range_volume_interaction'] = frame['intraday_range'] * frame['volume_zscore_20d']

custom_feature_names = [
    'mom_vol_ratio',
    'trend_spread',
    'quality_signal',
    'range_volume_interaction',
]

all_feature_names = base_feature_names + custom_feature_names
display(frame.loc[:, ['date', 'ticker'] + custom_feature_names].head())

,date,ticker,mom_vol_ratio,trend_spread,quality_signal,range_volume_interaction
0,2014-01-02,APA,NaN,NaN,NaN,NaN
1,2014-01-03,APA,NaN,NaN,NaN,NaN
2,2014-01-06,APA,NaN,NaN,NaN,NaN
3,2014-01-07,APA,NaN,NaN,NaN,NaN
4,2014-01-08,APA,NaN,NaN,NaN,NaN


## 4. Build Targets

We are going to fit three separate small MLPs with the exact same input features:

- one model for `expected_return`
- one model for `expected_alpha`
- one model for `expected_volatility`

This is not required, but it demonstrates the richer prediction contract supported by the toolkit.

Target builders used here:

- `make_forward_return_target(...)`
- `make_forward_alpha_target(...)`
- `make_forward_realized_vol_target(...)`

In plain English:
“What should the model learn to predict?”


In [18]:
return_targets = make_forward_return_target(prices, horizon=horizon)
alpha_targets = make_forward_alpha_target(prices, horizon=horizon)
vol_targets = make_forward_realized_vol_target(prices, window=horizon)

target_frame = frame.merge(return_targets, on=['date', 'ticker'], how='left')
target_frame = target_frame.merge(alpha_targets, on=['date', 'ticker'], how='left')
target_frame = target_frame.merge(vol_targets, on=['date', 'ticker'], how='left')

# Drop rows only after all features and targets are assembled.
# This is the usual notebook pattern because long-window features and forward targets
# naturally create missing values near the beginning and end of each ticker history.
target_frame = target_frame.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

return_target_col = f'forward_return_{horizon}d'
alpha_target_col = f'forward_alpha_{horizon}d_vs_spy'
vol_target_col = f'forward_realized_vol_{horizon}d'

print('Modeling frame shape after dropping nulls:', target_frame.shape)
display(target_frame.head())

Modeling frame shape after dropping nulls: (0, 21)


,date,ticker,return_5d,return_20d,vol_20d,momentum_20d,momentum_60d,rsi_14,price_to_sma_20d,price_to_sma_50d,...,excess_return_20d_vs_spy,intraday_range,beta_20d_spy,mom_vol_ratio,trend_spread,quality_signal,range_volume_interaction,forward_return_5d,forward_alpha_5d_vs_spy,forward_realized_vol_5d


## 5. Shared Train / Validation / Test Splits And Feature Scaling

This section does two very important things:

1. Uses the repo's shared split functions so the notebook respects the official date windows.
2. Standardizes features **using only the training split statistics**.

That second part matters a lot.

We do **not** want to normalize using future information from validation or test rows. So we compute mean and standard deviation from the train split only, then reuse those values everywhere else.

In plain English:
“Learn on past data, test on future data without cheating.”


In [19]:
train = slice_split(target_frame, dataset_name, "train", repo_root=repo_root)
val   = slice_split(target_frame, dataset_name, "val",   repo_root=repo_root)
test  = slice_split(target_frame, dataset_name, "test",  repo_root=repo_root)
 
print("Train rows:", len(train))
print("Val rows:",   len(val))
print("Test rows:",  len(test))
 
# Standardize using training-set statistics only.
train_means = train[all_feature_names].mean()
train_stds  = train[all_feature_names].std(ddof=0).replace(0.0, 1.0)
 
def standardize(feature_frame: pd.DataFrame) -> np.ndarray:
    return ((feature_frame[all_feature_names] - train_means) / train_stds).to_numpy(dtype=float)
 
X_train = standardize(train)
X_val   = standardize(val)
X_test  = standardize(test)
 
y_train_return = train[return_target_col].to_numpy(dtype=float)
y_val_return   = val[return_target_col].to_numpy(dtype=float)
y_test_return  = test[return_target_col].to_numpy(dtype=float)
 
y_train_alpha  = train[alpha_target_col].to_numpy(dtype=float)
y_val_alpha    = val[alpha_target_col].to_numpy(dtype=float)
 
y_train_vol    = train[vol_target_col].to_numpy(dtype=float)
y_val_vol      = val[vol_target_col].to_numpy(dtype=float)
 
print("X_train shape:", X_train.shape)
print("Feature count:", X_train.shape[1])

Train rows: 0
Val rows: 0
Test rows: 0
X_train shape: (0, 16)
Feature count: 16


## 6. Define A Very Basic MLP Class

This is a tiny PyTorch implementation of a feed-forward neural network for regression.

It is intentionally simple so the workflow is easy to understand:

- fully connected layers
- ReLU hidden activations
- linear output layer
- mean squared error loss
- mini-batch gradient descent
- validation loss tracking

This is **not** meant to be the most advanced PyTorch training setup. It is here so the notebook shows a real neural-network workflow while still staying readable.

In a real team workflow, this cell is exactly the part a researcher would replace with their own model implementation.

In plain English:
“This is the brain that learns patterns.”


In [10]:
#Define models
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
 
device = torch.device("mps" if torch.mps.is_available() else "cpu")
print("Using device:", device)
 
 
class Autoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim=LATENT_DIM):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim),
        )
 
    def forward(self, x):
        return self.decoder(self.encoder(x))
 
 
class TorchMLPRegressor(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )
 
    def forward(self, x):
        return self.net(x)

Using device: mps


## 7. Train Three Small MLPs

We are using the same feature matrix to train three related regressors:

- return model
- alpha model
- volatility model

This is a very common pattern in research projects:

- one common feature pipeline
- multiple prediction heads or target-specific models

The code below wraps the repeated steps in a helper function so the notebook stays readable.

In plain English:
“Teach the model what patterns lead to future returns.”


In [11]:
#train helpers
def train_autoencoder(X_tr, X_va):
    """Trains the autoencoder with minibatching and early stopping."""
    model     = Autoencoder(input_dim=X_tr.shape[1]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    loss_fn   = nn.MSELoss()
 
    X_tr_t = torch.tensor(X_tr, dtype=torch.float32).to(device)
    X_va_t = torch.tensor(X_va, dtype=torch.float32).to(device)
 
    loader = DataLoader(
        TensorDataset(X_tr_t, X_tr_t),
        batch_size=BATCH_SIZE,
        shuffle=True,
    )
 
    best_val_loss = float("inf")
    patience_counter = 0
 
    for epoch in range(AE_EPOCHS):
        model.train()
        epoch_loss = 0.0
        for xb, _ in loader:
            optimizer.zero_grad()
            loss = loss_fn(model(xb), xb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(xb)
        epoch_loss /= len(X_tr)
 
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_va_t), X_va_t).item()
 
        print(f"AE  Epoch {epoch+1:3d}/{AE_EPOCHS} | train loss: {epoch_loss:.6f} | val loss: {val_loss:.6f}")
 
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"  Early stopping at epoch {epoch+1} (no val improvement for {PATIENCE} epochs).")
                break
 
    model.load_state_dict(best_state)
    return model
 
 
def train_predictor(X_tr, y_tr, X_va, y_va, label="MLP"):
    """Trains a downstream MLP predictor with minibatching and early stopping."""
    model     = TorchMLPRegressor(input_dim=X_tr.shape[1]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    loss_fn   = nn.MSELoss()
 
    X_tr_t = torch.tensor(X_tr, dtype=torch.float32).to(device)
    y_tr_t = torch.tensor(y_tr.reshape(-1, 1), dtype=torch.float32).to(device)
    X_va_t = torch.tensor(X_va, dtype=torch.float32).to(device)
    y_va_np = y_va.reshape(-1)
 
    loader = DataLoader(
        TensorDataset(X_tr_t, y_tr_t),
        batch_size=BATCH_SIZE,
        shuffle=True,
    )
 
    best_val_loss = float("inf")
    patience_counter = 0
 
    for epoch in range(MLP_EPOCHS):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()
            loss_fn(model(xb), yb).backward()
            optimizer.step()
 
        model.eval()
        with torch.no_grad():
            val_pred = model(X_va_t).cpu().numpy().reshape(-1)
        val_loss = np.mean((val_pred - y_va_np) ** 2)
 
        print(f"{label} Epoch {epoch+1:3d}/{MLP_EPOCHS} | val MSE: {val_loss:.6f}")
 
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"  Early stopping at epoch {epoch+1}.")
                break
 
    model.load_state_dict(best_state)
    return model
 
 
def encode(ae_model, X):
    """Extract latent representations from a trained autoencoder encoder."""
    ae_model.eval()
    with torch.no_grad():
        return ae_model.encoder(torch.tensor(X, dtype=torch.float32).to(device)).cpu().numpy()
 
 
def predict(model, X):
    """Run inference with a trained predictor."""
    model.eval()
    with torch.no_grad():
        return model(torch.tensor(X, dtype=torch.float32).to(device)).cpu().numpy().reshape(-1)
    
#train autoencoder
autoencoder = train_autoencoder(X_train, X_val)
 
Z_train = encode(autoencoder, X_train)
Z_val   = encode(autoencoder, X_val)
Z_test  = encode(autoencoder, X_test)
 


#train downstream predictors
print("--- Training return predictor ---")
return_model = train_predictor(Z_train, y_train_return, Z_val, y_val_return, label="Return MLP")
 
print("\n--- Training alpha predictor ---")
alpha_model  = train_predictor(Z_train, y_train_alpha,  Z_val, y_val_alpha,  label="Alpha  MLP")

AE  Epoch   1/25 | train loss: 0.774874 | val loss: 0.665340
AE  Epoch   2/25 | train loss: 0.332426 | val loss: 0.329314
AE  Epoch   3/25 | train loss: 0.166747 | val loss: 0.196088
AE  Epoch   4/25 | train loss: 0.102161 | val loss: 0.129666
AE  Epoch   5/25 | train loss: 0.070586 | val loss: 0.109613
AE  Epoch   6/25 | train loss: 0.058001 | val loss: 0.094483
AE  Epoch   7/25 | train loss: 0.048290 | val loss: 0.079161
AE  Epoch   8/25 | train loss: 0.039450 | val loss: 0.066152
AE  Epoch   9/25 | train loss: 0.032102 | val loss: 0.055410
AE  Epoch  10/25 | train loss: 0.025525 | val loss: 0.046136
AE  Epoch  11/25 | train loss: 0.019690 | val loss: 0.037869
AE  Epoch  12/25 | train loss: 0.015465 | val loss: 0.030516
AE  Epoch  13/25 | train loss: 0.012599 | val loss: 0.024475
AE  Epoch  14/25 | train loss: 0.010822 | val loss: 0.019655
AE  Epoch  15/25 | train loss: 0.009683 | val loss: 0.017518
AE  Epoch  16/25 | train loss: 0.008852 | val loss: 0.015765
AE  Epoch  17/25 | train

## 8. Create The Standardized Prediction Table

Now we convert raw model outputs into the shared prediction contract used by the rest of the toolkit.

Required columns:

- `date`
- `ticker`
- `horizon`
- `expected_return`

Optional columns we will also populate:

- `expected_alpha`
- `expected_volatility`
- `uncertainty`

For this notebook, `uncertainty` is just a simple constant based on validation RMSE for the return model. That is not a sophisticated uncertainty estimate. It is only here to demonstrate where that information would live in the shared schema.

In plain English:
“For each stock, what do we think will happen next?”


In [20]:
# HOW IT WORKS:
#   1. Compute each ticker's daily log return from the shared price frame.
#   2. Roll a 20-day std over those returns.
#   3. Annualize by multiplying by sqrt(252).
#   4. Join onto the test set by (date, ticker).
#   5. Fall back to the cross-sectional mean for any missing values.
# =============================================================================
 
# --- Step 1-3: compute rolling realized vol per ticker ---
prices_sorted = prices.sort_values(["ticker", "date"]).copy()
prices_sorted["log_return"] = (
    prices_sorted
    .groupby("ticker")["close"]          # use whichever price column your toolkit provides
    .transform(lambda s: np.log(s).diff())
)
prices_sorted["realized_vol_20d"] = (
    prices_sorted
    .groupby("ticker")["log_return"]
    .transform(lambda s: s.rolling(20, min_periods=10).std() * np.sqrt(252))
)
 
vol_lookup = prices_sorted[["date", "ticker", "realized_vol_20d"]].dropna()
 
# --- Step 4: join onto test rows ---
test_with_vol = test[["date", "ticker"]].merge(vol_lookup, on=["date", "ticker"], how="left")
 
# --- Step 5: fill any missing with cross-sectional mean ---
fallback_vol = vol_lookup["realized_vol_20d"].mean()
realized_vol_test = test_with_vol["realized_vol_20d"].fillna(fallback_vol).to_numpy()
 
print(f"Realized vol stats (test set):")
print(f"  mean: {realized_vol_test.mean():.4f}")
print(f"  std:  {realized_vol_test.std():.4f}")
print(f"  min:  {realized_vol_test.min():.4f}")
print(f"  max:  {realized_vol_test.max():.4f}")
print(f"  missing filled with fallback: {test_with_vol['realized_vol_20d'].isna().sum()} rows")
 
# Sanity check — vol and |return| should now be on similar scales
test_return_pred = predict(return_model, Z_test)
print(f"\nScale check:")
print(f"  |expected_return| mean: {np.abs(test_return_pred).mean():.4f}")
print(f"  realized_vol mean:      {realized_vol_test.mean():.4f}")
print(f"  ratio (vol / |ret|):    {realized_vol_test.mean() / np.abs(test_return_pred).mean():.1f}x  (target: < 20x)")
 
# --- Build prediction frame ---
test_alpha_pred = predict(alpha_model, Z_test)
 
predictions = test.loc[:, ["date", "ticker"]].copy()
predictions["horizon"]             = horizon
predictions["expected_return"]     = test_return_pred
predictions["expected_alpha"]      = test_alpha_pred
predictions["expected_volatility"] = np.clip(realized_vol_test, 1e-4, None)
predictions["uncertainty"]         = np.abs(test_return_pred - test_return_pred.mean())
 
predictions = validate_prediction_frame(
    predictions,
    dataset_name='secret_set_1',
    horizon=horizon,
    repo_root=repo_root,
)
display(predictions.head())
 

Realized vol stats (test set):
  mean: nan
  std:  nan


/var/folders/3y/dhkqtqns7p35w8t4_svmjz780000gn/T/ipykernel_36823/3960697257.py:32: RuntimeWarning: Mean of empty slice
  print(f"  mean: {realized_vol_test.mean():.4f}")
/Users/adamthorne/.pyenv/versions/3.12.7/lib/python3.12/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/Users/adamthorne/.pyenv/versions/3.12.7/lib/python3.12/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/adamthorne/.pyenv/versions/3.12.7/lib/python3.12/site-packages/numpy/_core/_methods.py:178: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/Users/adamthorne/.pyenv/versions/3.12.7/lib/python3.12/site-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


ValueError: zero-size array to reduction operation minimum which has no identity

## 9. Turn Predictions Into A Portfolio Object

The toolkit separates forecasting from portfolio construction.

Here we use the built-in `weights_from_predictions_risk_adjusted(...)` helper.

What it does:

- uses `expected_return / expected_volatility` as the score
- keeps the allocation long-only
- normalizes the scores so each row sums to `1.0`
- returns a `PortfolioWeights` object

This is a good default for demonstrations because it uses more of the prediction contract than a plain top-k rule.

In plain English:
“Use predictions to decide how to allocate money.”


In [12]:
portfolio = weights_from_predictions_risk_adjusted(
    predictions,
    dataset_name='secret_set_1',
    strategy_name=model_name,
)
validated_weights = validate_weights_frame(
    portfolio.weights,
    dataset_name='secret_set_1',
    repo_root=repo_root,
)
print("Strategy name:",        portfolio.strategy_name)
print("Weights frame shape:",  validated_weights.shape)
display(validated_weights.head())

NameError: name 'predictions' is not defined

## 10. Run The Shared Backtest

This is where the toolkit gives you the most value as shared infrastructure.

The backtest layer will:

- load the shared dataset prices
- align rebalance dates to the next available trading day
- apply transaction costs from the dataset preset
- compare the strategy to benchmarks like `SPY`
- compute NAV, returns, turnover, and summary metrics

Because this is shared across the team, different notebooks remain comparable even if the model logic is very different.

In plain English:
“If we followed this strategy in the past, how much money would we make?”


In [ ]:
result       = backtest_weights(dataset_name, portfolio, repo_root=repo_root)
metrics      = build_metrics(result)
artifact_paths = write_backtest_artifacts(result, output_dir)
 
metrics_table = (
    pd.DataFrame([{"metric": k, "value": v} for k, v in sorted(metrics.items())])
    .sort_values("metric")
    .reset_index(drop=True)
)
display(metrics_table)
print("QuantStats report:", artifact_paths["quantstats_report"])

## 11. Log The Run To MLflow

The toolkit keeps MLflow intentionally lightweight:

- local SQLite backend
- local artifact storage
- notebook-friendly logging helpers

The pattern here is the one you want teammates to reuse:

1. initialize MLflow locally
2. start a run with meaningful tags
3. log predictions, portfolio weights, and backtest results
4. let MLflow keep the artifact trail

In plain English:
“Record everything so we can compare runs later.”


In [ ]:
mlflow_layout = init_mlflow(repo_root)
print("MLflow tracking URI:", mlflow_layout["tracking_uri"])
 
with start_run(
    run_name="Hannah_Init",
    dataset_name=dataset_name,
    tags={
        "workflow":           "autoencoder_downstream",
        "model_family":       "autoencoder",
        "prediction_horizon": str(horizon),
    },
    repo_root=repo_root,
):
    import mlflow
 
    mlflow.log_params({
        "model_name":          model_name,
        "dataset_name":        dataset_name,
        "horizon":             horizon,
        "feature_count":       len(all_feature_names),
        "base_feature_list":   ",".join(base_feature_names),
        "custom_feature_list": ",".join(custom_feature_names),
        "latent_dim":          LATENT_DIM,
        "ae_epochs":           AE_EPOCHS,        # was: 'epochs': 35 (wrong)
        "mlp_epochs":          MLP_EPOCHS,        # was: missing
        "batch_size":          BATCH_SIZE,
        "learning_rate":       LEARNING_RATE,
        "early_stopping_patience": PATIENCE,
        "ae_hidden_dims":      "64",
        "mlp_hidden_dims":     "32",
        "portfolio_builder":   "weights_from_predictions_risk_adjusted",
        'random_seed': 99,
        "cost_bps":            spec.cost_bps,
    })
 
    log_predictions(predictions)
    log_portfolio(portfolio)
    log_backtest(result)
    print("MLflow logging complete.")

## 12. Inspect Results

At this point the notebook has produced:

- validated predictions
- validated portfolio weights
- a shared backtest result
- standardized performance metrics
- a QuantStats HTML report
- an MLflow run with artifacts and metrics

That is the full intended research loop for this repo.

In plain English:
“Did this strategy actually work?”


In [ ]:
print("Top-level metrics:")
for key, value in sorted(result.metrics.items()):
    print(f"  {key}: {value:.6f}")
 
print("\nArtifact paths:")
for key, value in artifact_paths.items():
    print(f"  {key}: {value}")
 
display(result.nav.tail().to_frame("nav"))
display(result.returns.tail().to_frame("returns"))
display(result.turnover.tail().to_frame("turnover"))

In [ ]:
# ---------------------------------------------------------------------
# Final validation cell.
# ---------------------------------------------------------------------
# These checks are intentionally simple. They are the kind of sanity checks
# you want at the end of a notebook before you trust the output.

# In plain English:
# “Make sure nothing is broken before calling it done.”

# ---------------------------------------------------------------------

assert {"total_return", "annual_return", "sharpe", "max_drawdown"}.issubset(result.metrics)
assert validated_weights.index.is_monotonic_increasing
assert (validated_weights.sum(axis=1).round(6) == 1.0).all()
assert Path(artifact_paths["quantstats_report"]).exists()
assert {"date", "ticker", "horizon", "expected_return"}.issubset(predictions.columns)
assert predictions["uncertainty"].nunique() > 1, "uncertainty must be per-asset, not a scalar"
assert predictions["expected_alpha"].nunique() > 1, "expected_alpha must differ from expected_return"
print("End-to-end autoencoder workflow validated successfully.")

## 13. Backtest Hannah's trained model on `secret_set_1`

This section assumes Hannah's autoencoder and downstream predictors above are already trained in the current kernel. It does not retrain or modify the model weights.

Hannah's feature contract was trained with SPY-relative feature names. For the secret backtest, SPY is loaded only as an inference-time feature-engineering context through a one-ticker custom dataset. SPY is merged into the feature frame, then removed before prediction validation and portfolio construction. The backtest itself uses the secret dataset benchmark, `XLE`.


In [30]:
from pathlib import Path
import numpy as np
import pandas as pd
from portfolio_toolkit import custom_dataset

secret_dataset_name = "secret_set_1"
secret_feature_benchmark = "SPY"
secret_rebalance_frequency = "daily"  # choose: "daily", "weekly", or "monthly"
hannah_training_dataset_name = "shared_set_2"  # used only to rebuild preprocessing stats if the in-memory stats were clobbered

repo_root = Path(repo_root).resolve() if "repo_root" in globals() else Path.cwd().resolve()
if not (repo_root / "configs" / "datasets.toml").exists():
    repo_root = Path.cwd().resolve()
if not (repo_root / "configs" / "datasets.toml").exists():
    raise FileNotFoundError(f"Could not find configs/datasets.toml from repo_root={repo_root}")

required_trained_objects = [
    "autoencoder",
    "return_model",
    "alpha_model",
    "encode",
    "predict",
    "all_feature_names",
    "base_feature_names",
    "custom_feature_names",
    "model_name",
    "horizon",
]
missing_trained_objects = [name for name in required_trained_objects if name not in globals()]
if missing_trained_objects:
    raise NameError(
        "Train Hannah's notebook model first. Missing objects: "
        + ", ".join(missing_trained_objects)
    )

secret_spec = get_dataset_spec(secret_dataset_name, repo_root=repo_root)
secret_prices = load_prices(secret_dataset_name, repo_root=repo_root)
secret_backtest_benchmark = secret_spec.default_benchmark

spy_feature_dataset = custom_dataset(
    [secret_feature_benchmark],
    start=secret_spec.start_date,
    end=secret_spec.end_date,
    benchmark=secret_feature_benchmark,
    name="hannah_spy_feature_context",
    cost_bps=secret_spec.cost_bps,
)
spy_feature_prices = load_prices(spy_feature_dataset, repo_root=repo_root)

secret_feature_prices = (
    pd.concat([secret_prices, spy_feature_prices], ignore_index=True)
    .drop_duplicates(["date", "ticker"], keep="last")
    .sort_values(["ticker", "date"])
    .reset_index(drop=True)
)

print("Secret dataset:", secret_dataset_name)
print("Display name:", secret_spec.name)
print("Feature benchmark ticker:", secret_feature_benchmark)
print("Backtest benchmark ticker:", secret_backtest_benchmark)
print("Modeled tickers:", len(secret_spec.tickers))
print("Feature-frame tickers:", sorted(secret_feature_prices["ticker"].unique()))
print("Rebalance frequency:", secret_rebalance_frequency)


Secret dataset: secret_set_1
Display name: XLE_test
Feature benchmark ticker: SPY
Backtest benchmark ticker: XLE
Modeled tickers: 22
Feature-frame tickers: ['APA', 'BKR', 'COP', 'CTRA', 'CVX', 'DVN', 'EOG', 'EQT', 'EXE', 'FANG', 'HAL', 'KMI', 'MPC', 'OKE', 'OXY', 'PSX', 'SLB', 'SPY', 'TPL', 'TRGP', 'VLO', 'WMB', 'XLE', 'XOM']
Rebalance frequency: daily


In [31]:
def select_rebalance_dates(dates, frequency: str) -> pd.DatetimeIndex:
    dates = pd.DatetimeIndex(pd.to_datetime(sorted(pd.unique(dates)))).tz_localize(None)
    frequency = frequency.strip().lower()
    if frequency == "daily":
        return dates
    if frequency == "weekly":
        selected = dates.to_series(index=dates).groupby(dates.to_period("W-FRI")).first()
        return pd.DatetimeIndex(selected.to_numpy())
    if frequency == "monthly":
        selected = dates.to_series(index=dates).groupby(dates.to_period("M")).first()
        return pd.DatetimeIndex(selected.to_numpy())
    raise ValueError("secret_rebalance_frequency must be one of: 'daily', 'weekly', 'monthly'")


def build_hannah_feature_frame(prices_with_spy: pd.DataFrame) -> pd.DataFrame:
    return build_features(prices_with_spy, feature_names=base_feature_names).copy()


def add_hannah_custom_features(feature_frame: pd.DataFrame) -> pd.DataFrame:
    frame = feature_frame.copy()
    eps = 1e-6
    frame["mom_vol_ratio"] = frame["momentum_20d"] / (frame["vol_20d"].abs() + eps)
    frame["trend_spread"] = frame["price_to_sma_20d"] - frame["price_to_sma_50d"]
    frame["quality_signal"] = frame["excess_return_20d_vs_spy"] - 0.5 * frame["vol_20d"]
    frame["range_volume_interaction"] = frame["intraday_range"] * frame["volume_zscore_20d"]
    return frame


In [32]:
def preprocessing_stats_are_usable(means, stds, feature_names) -> bool:
    if means is None or stds is None:
        return False
    means = pd.Series(means).reindex(feature_names)
    stds = pd.Series(stds).reindex(feature_names)
    return bool(means.notna().all() and stds.notna().all() and np.isfinite(stds.replace(0.0, 1.0)).all())


def rebuild_hannah_preprocessing_stats(training_dataset_name: str) -> tuple[pd.Series, pd.Series]:
    training_prices = load_prices(training_dataset_name, repo_root=repo_root)
    training_features = add_hannah_custom_features(build_hannah_feature_frame(training_prices))
    training_features = training_features.replace([np.inf, -np.inf], np.nan).dropna(subset=all_feature_names)
    training_split = slice_split(training_features, training_dataset_name, "train", repo_root=repo_root)
    if training_split.empty:
        raise ValueError(f"Could not rebuild preprocessing stats; {training_dataset_name} train split is empty")
    means = training_split[all_feature_names].mean()
    stds = training_split[all_feature_names].std(ddof=0).replace(0.0, 1.0)
    return means, stds


if preprocessing_stats_are_usable(globals().get("train_means"), globals().get("train_stds"), all_feature_names):
    secret_train_means = pd.Series(train_means).reindex(all_feature_names)
    secret_train_stds = pd.Series(train_stds).reindex(all_feature_names).replace(0.0, 1.0)
    secret_preprocessing_stats_source = "current_kernel"
else:
    secret_train_means, secret_train_stds = rebuild_hannah_preprocessing_stats(hannah_training_dataset_name)
    secret_preprocessing_stats_source = f"rebuilt_from_{hannah_training_dataset_name}"

print("Preprocessing stats source:", secret_preprocessing_stats_source)
print("Feature count:", len(all_feature_names))


Preprocessing stats source: rebuilt_from_shared_set_2
Feature count: 16


In [33]:
secret_features_with_spy = add_hannah_custom_features(build_hannah_feature_frame(secret_feature_prices))
missing_secret_features = [feature for feature in all_feature_names if feature not in secret_features_with_spy.columns]
if missing_secret_features:
    raise ValueError("Secret feature frame is missing: " + ", ".join(missing_secret_features))

secret_model_frame = secret_features_with_spy.loc[
    secret_features_with_spy["ticker"].isin(secret_spec.tickers)
].copy()
secret_model_frame = slice_split(secret_model_frame, secret_dataset_name, "test", repo_root=repo_root)
secret_model_frame = secret_model_frame.replace([np.inf, -np.inf], np.nan).dropna(subset=all_feature_names)

secret_test_dates = pd.DatetimeIndex(sorted(secret_model_frame["date"].unique()))
secret_rebalance_dates = select_rebalance_dates(secret_test_dates, secret_rebalance_frequency)
secret_model_frame = secret_model_frame.loc[secret_model_frame["date"].isin(secret_rebalance_dates)].copy()

if secret_model_frame.empty:
    raise ValueError("Secret inference frame is empty after feature filtering and rebalance selection")

secret_X = ((secret_model_frame[all_feature_names] - secret_train_means) / secret_train_stds).to_numpy(dtype=float)
secret_Z = encode(autoencoder, secret_X)

print("Secret inference rows:", len(secret_model_frame))
print("Secret rebalance dates:", len(secret_rebalance_dates), secret_rebalance_dates.min(), "to", secret_rebalance_dates.max())
print("Secret X shape:", secret_X.shape)
print("Secret Z shape:", secret_Z.shape)
print("Prediction tickers include SPY:", "SPY" in set(secret_model_frame["ticker"]))


Secret inference rows: 22063
Secret rebalance dates: 1003 2022-01-03 00:00:00 to 2025-12-31 00:00:00
Secret X shape: (22063, 16)
Secret Z shape: (22063, 16)
Prediction tickers include SPY: False


In [34]:
secret_prices_sorted = secret_prices.sort_values(["ticker", "date"]).copy()
secret_prices_sorted["log_return"] = (
    secret_prices_sorted.groupby("ticker")["close"].transform(lambda s: np.log(s).diff())
)
secret_prices_sorted["realized_vol_20d"] = (
    secret_prices_sorted.groupby("ticker")["log_return"]
    .transform(lambda s: s.rolling(20, min_periods=10).std() * np.sqrt(252))
)
secret_vol_lookup = secret_prices_sorted[["date", "ticker", "realized_vol_20d"]].dropna()
secret_test_with_vol = secret_model_frame[["date", "ticker"]].merge(
    secret_vol_lookup,
    on=["date", "ticker"],
    how="left",
)
secret_fallback_vol = secret_vol_lookup["realized_vol_20d"].mean()
if not np.isfinite(secret_fallback_vol):
    raise ValueError("Could not compute fallback realized volatility for secret predictions")
secret_realized_vol_test = secret_test_with_vol["realized_vol_20d"].fillna(secret_fallback_vol).to_numpy()

secret_return_pred = predict(return_model, secret_Z)
secret_alpha_pred = predict(alpha_model, secret_Z)

secret_predictions = secret_model_frame.loc[:, ["date", "ticker"]].copy()
secret_predictions["horizon"] = horizon
secret_predictions["expected_return"] = secret_return_pred
secret_predictions["expected_alpha"] = secret_alpha_pred
secret_predictions["expected_volatility"] = np.clip(secret_realized_vol_test, 1e-4, None)
secret_predictions["uncertainty"] = np.abs(secret_return_pred - secret_return_pred.mean())

secret_predictions = validate_prediction_frame(
    secret_predictions,
    dataset_name=secret_dataset_name,
    horizon=horizon,
    repo_root=repo_root,
)

print("Secret predictions shape:", secret_predictions.shape)
display(secret_predictions.head())


Secret predictions shape: (22063, 7)


,date,ticker,horizon,expected_return,expected_alpha,expected_volatility,uncertainty
0,2022-01-03,APA,5,0.013791,0.014929,0.484839,0.004067
1,2022-01-03,BKR,5,0.008412,0.022643,0.323390,0.001313
2,2022-01-03,COP,5,0.010091,0.011457,0.311513,0.000366
3,2022-01-03,CTRA,5,0.010077,0.020509,0.317424,0.000353
4,2022-01-03,CVX,5,0.005187,0.002691,0.188475,0.004537


In [35]:
secret_portfolio = weights_from_predictions_risk_adjusted(
    secret_predictions,
    dataset_name=secret_dataset_name,
    strategy_name=f"{model_name}_{secret_dataset_name}_{secret_rebalance_frequency}",
)
secret_validated_weights = validate_weights_frame(
    secret_portfolio.weights,
    dataset_name=secret_dataset_name,
    repo_root=repo_root,
)

secret_result = backtest_weights(
    secret_dataset_name,
    secret_portfolio,
    benchmark=secret_backtest_benchmark,
    repo_root=repo_root,
)
secret_metrics = build_metrics(secret_result)
secret_output_dir = repo_root / "runs" / "hannah_secret_backtest"
secret_output_dir.mkdir(parents=True, exist_ok=True)
secret_artifact_paths = write_backtest_artifacts(secret_result, secret_output_dir)

secret_metrics_table = (
    pd.DataFrame([{"metric": key, "value": value} for key, value in sorted(secret_metrics.items())])
    .sort_values("metric")
    .reset_index(drop=True)
)

print("Secret strategy name:", secret_portfolio.strategy_name)
print("Secret weights shape:", secret_validated_weights.shape)
print("Backtest benchmark requested:", secret_backtest_benchmark)
display(secret_metrics_table)
print("Secret QuantStats report:", secret_artifact_paths["quantstats_report"])


Secret strategy name: autoencoder_mlp_downstream_secret_set_1_daily
Secret weights shape: (1003, 22)
Backtest benchmark requested: XLE


,metric,value
0,annual_excess_return_vs_benchmark,-0.120395
1,annual_return,0.038527
2,annual_volatility,0.274343
3,average_turnover,0.244683
4,benchmark_annual_return,0.158922
5,benchmark_annual_volatility,0.258439
6,benchmark_max_drawdown,-0.260437
7,benchmark_sharpe,0.614931
8,benchmark_total_return,0.801734
9,calmar,0.127874


Secret QuantStats report: /Users/adamthorne/Desktop/Portfolio/Portfolio-Optimizer/runs/hannah_secret_backtest/quantstats.html


In [37]:
import mlflow
from mlflow.tracking import MlflowClient

mlflow_layout = init_mlflow(repo_root)
mlflow.set_tracking_uri(mlflow_layout["tracking_uri"])

client = MlflowClient()
client.restore_experiment("0")


In [38]:
secret_mlflow_layout = init_mlflow(repo_root)
print("MLflow tracking URI:", secret_mlflow_layout["tracking_uri"])

with start_run(
    run_name="Hannah_Init_secret_backtest",
    dataset_name=secret_dataset_name,
    tags={
        "workflow": "hannah_secret_backtest",
        "model_family": "autoencoder",
        "prediction_horizon": str(horizon),
        "rebalance_frequency": secret_rebalance_frequency,
        "feature_benchmark": secret_feature_benchmark,
    },
    repo_root=repo_root,
):
    import mlflow

    mlflow.log_params({
        "model_name": model_name,
        "dataset_name": secret_dataset_name,
        "horizon": horizon,
        "feature_count": len(all_feature_names),
        "base_feature_list": ",".join(base_feature_names),
        "custom_feature_list": ",".join(custom_feature_names),
        "latent_dim": LATENT_DIM,
        "portfolio_builder": "weights_from_predictions_risk_adjusted",
        "rebalance_frequency": secret_rebalance_frequency,
        "feature_benchmark_ticker": secret_feature_benchmark,
        "feature_context_dataset": spy_feature_dataset.identifier,
        "preprocessing_stats_source": secret_preprocessing_stats_source,
        "feature_compatibility_note": "SPY merged for feature engineering only; SPY excluded from secret predictions and weights",
    })

    log_predictions(secret_predictions)
    log_portfolio(secret_portfolio)
    log_backtest(secret_result)
    print("Secret-set MLflow logging complete.")


MLflow tracking URI: https://adams-macbook-pro.tail5ddc35.ts.net
Secret-set MLflow logging complete.
🏃 View run Hannah_Init_secret_backtest at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/3/runs/58a7e7ab3bea44aabf343a9755c37c3c
🧪 View experiment at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/3


In [ ]:
assert not secret_predictions.empty
assert not secret_validated_weights.empty
assert {"total_return", "annual_return", "sharpe", "max_drawdown"}.issubset(secret_result.metrics)
assert secret_validated_weights.index.is_monotonic_increasing
assert (secret_validated_weights.sum(axis=1).round(6) == 1.0).all()
assert Path(secret_artifact_paths["quantstats_report"]).exists()
print("Secret-set backtest cells validated successfully.")
